In [ ]:
using Flux
using CUDA
using FileIO
using Images
using ImageMagick  # Ensure ImageMagick is loaded

dev=gpu

T=Float64

function load_images_from_folder(folder_path)
    images = []
    supported_extensions = [".jpg", ".jpeg", ".png", ".bmp", ".gif"]
    for file in readdir(folder_path)
        ext = lowercase(splitext(file)[2])
        if ext in supported_extensions
            img_path = joinpath(folder_path, file)
            img = load(img_path)
            push!(images, img)
        end
    end
    return images
end
function preprocess_images(images, target_size=(64, 64))
    return [imresize(img, target_size) ./ 255.0 for img in images]
end
human_images_path = "C:\\Users\\jonat\\OneDrive\\Desktop\\Human_vs_non_human\\human-and-non-human\\versions\\1\\human-and-non-human\\training_set\\training_set\\humans"
non_human_images_path = "C:\\Users\\jonat\\OneDrive\\Desktop\\Human_vs_non_human\\human-and-non-human\\versions\\1\\human-and-non-human\\training_set\\training_set\\non-humans"

human_images = load_images_from_folder(human_images_path)
non_human_images = load_images_from_folder(non_human_images_path)


human_images = preprocess_images(human_images)
non_human_images = preprocess_images(non_human_images)
 
typeof(human_images)


human_labels = [1 for _ in 1:length(human_images)]
non_human_labels = [0 for _ in 1:length(non_human_images)]

images = vcat(human_images, non_human_images)
labels = vcat(human_labels, non_human_labels)

#print(typeof(images))
data = [(images[i], labels[i]) for i in 1:length(images)]
print("here")


using Flux: OneHotMatrix
x_train = Flux.flatten(images)
x_train2 = Flux.flatten(x_train)
print(typeof(xtrain2))

x_test  = Flux.flatten(x_test)

y_train = Flux.onehotbatch(labels, 0:1)


In [54]:
using Random
using RobustNeuralNetworks


rng = MersenneTwister(42)

nu=64*64*3
ny=2
nh = [128, 64]     # Two hidden layers with 128 and 64 neurons respectively
γ  = 5.0f0  # Lipschitz bound of 5.0


model_ps = DenseLBDNParams{Float64}(nu, nh, ny, γ; rng)
model = Chain(DiffLBDN(model_ps), Flux.softmax) |> gpu  # Move model to G



Chain(
  DiffLBDN(
    DenseLBDNParams(
      DirectLBDNParams{Float32, 3, 2}((Float32[0.015282118 -0.012408989 … -0.0083517935 0.004353685; -0.000997508 -0.03162503 … -0.018839177 -0.010980508; … ; 0.011214569 -0.006406235 … -0.015174186 -0.0147820795; -0.0014321277 -0.004880946 … -0.003841042 -0.016281005], Float32[0.07760339 -0.042777702 … -0.034165677 -0.16345395; 0.040468723 0.067645766 … -0.018089628 -0.07592612; … ; 0.11835045 -0.045802 … -0.15762645 -0.0012775161; -0.039567836 0.026907396 … 0.09381783 0.06353916], Float32[0.36482942 0.20072849; -0.16067915 0.15724745; … ; 0.3393401 -0.22236769; -0.055035718 0.099062465]), (Float32[15.912413], Float32[9.87965], Float32[2.0025327]), (Float32[0.075818636, -0.042531963, 0.121054016, 0.13886173, -0.07674013, -0.20824714, 0.11363562, -0.02135459, 0.16059129, -0.18726929  …  0.18917146, -0.08314908, -0.028080123, 0.25442967, -0.14990641, -0.030063309, 0.054680385, 0.0030262887, 0.22808073, 0.08654746], Float32[-0.055570554, 0.27633306

In [67]:
using Flux.Optimise: ADAM
using Flux.Losses: logitcrossentropy

optimizer = ADAM()
loss(x, y) = logitcrossentropy(model(x), y)



UndefVarError: UndefVarError: `ADAM` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Hint: a global variable of this name may be made accessible by importing Optimisers in the current active module Main

In [ ]:
using Statistics

# Check test accuracy during training
compare(y::OneHotMatrix, ŷ) = maximum(ŷ, dims=1) .== maximum(y.*ŷ, dims=1)
accuracy(model, x, y::OneHotMatrix) = mean(compare(y, model(x)))

# Callback function to show results while training
function progress(model, iter)
    train_loss = round(loss(model, x_train, y_train), digits=4)
    test_acc = round(accuracy(model, x_test, y_test), digits=4)
    @show iter train_loss test_acc
    println()
end
